In [2]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch.nn.functional as F
from torchinfo import summary

### Attention Examples

In [3]:
# -- Parameters
batch_size = 128
channels = 64
size = 32

x_test = torch.randn(batch_size, channels, size, size)

# -- Test inputs with attention heads
x = x_test.view(-1, channels, size * size).swapaxes(1, 2)
print(f"x shape: {x.shape}")
print(x_test.view(-1, channels, size * size).shape)
# x_ln = self.ln(x)



# mha = nn.MultiheadAttention(channels, 4, batch_first=True)
# x_mha_out = mha(x)

# print(f"x_mha_out shape: {x_mha_out.shape}")

x shape: torch.Size([128, 1024, 64])
torch.Size([128, 64, 1024])


### Attention blocks

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, channels, size):
        super(SelfAttention, self).__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, 4, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)

